# 📏 09 · How good is a map? FSC, half-maps, local resolution and particle selection
### Resolution is measured, not seen — and the measurement itself can be fooled

**BiologyToResearchV3** · Part B — Imaging physics · Module 09 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/09_resolution_validation.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed. Uses 2 CPU cores in Section 7. |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, `concurrent.futures` (standard library) |
| **Prerequisites** | [03 · Fourier transform](03_fourier_frequency.ipynb), [04 · Noise & statistics](04_noise_statistics_inference.ipynb), [06 · Image formation](06_electron_image_formation.ipynb) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Lü *et al.* 2020 (Spark-parallel local FSC, BMC Bioinformatics); Shi *et al.* 2020 (simulated-annealing image selection, Quantitative Biology); preview of the SFSC score of Xu *et al.* 2019 (Structure) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this notebook are small, controlled *toy versions* of published methods. They illustrate the core
> idea of each paper; they do **not** reproduce the papers' numbers. We use a real single-particle cryo-EM map of
> β-galactosidase distributed with the NGL viewer's test data; if you are offline, the notebook falls back to a simple
> stand-in shape and says so.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Convert** between spatial frequency (1/Å), Fourier shells and resolution (Å), and explain what "a 4 Å map" means.
2. **Implement** the Fourier shell correlation (FSC) between two independent half-maps and **read** a resolution at the
   0.143 (half-maps) and 0.5 (map vs. a known truth) thresholds — and explain why the two agree.
3. **Demonstrate** two ways FSC can lie: sharp masks and non-independent half-maps ("Einstein from noise").
4. **Derive and verify** the relation between the spectral SNR (SSNR) and FSC: $\text{FSC}_{\text{half}}=\text{SSNR}_{\text{half}}/(1+\text{SSNR}_{\text{half}})$,
   and its full-set form $\rho=\eta/(2+\eta)$ used by Multi-Pattern Pursuit and by Shi *et al.*
5. **Measure** how resolution grows with the number of particles, and compute a **local-resolution** map with a sliding
   window — in parallel, in the spirit of Lü *et al.*'s Spark implementation.
6. **Implement** resolution-guided particle selection by **simulated annealing** (Shi *et al.* 2020) with the standard
   Metropolis rule, and evaluate it with precision, recall and $F_{0.5}$.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load tools and the β-galactosidase map | 5 min |
| 1 · Resolution is a frequency | Å ↔ 1/Å ↔ Fourier shells; what blurring removes | 10 min |
| 2 · Half-maps and the FSC | build half-maps, compute FSC, thresholds 0.143 and 0.5 | 20 min |
| 3 · How FSC can lie | masks; overfitting noise ("Einstein from noise") | 20 min |
| 4 · SSNR and FSC | the formulas behind MPP's SFSC and Shi's score | 15 min |
| 5 · More particles, better maps? | FSC vs N, the Rosenthal–Henderson plot | 10 min |
| 6 · Local resolution | sliding-window FSC on β-galactosidase; a real local-resolution map | 15 min |
| 7 · Parallel local FSC | Spark's design, emulated with `concurrent.futures` — Lü *et al.* 2020 | 10 min |
| 8 · Choosing good particles | simulated-annealing selection — Shi *et al.* 2020 | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A cryo-EM paper announces "the structure of β-galactosidase at 3.2 Å". What does that number mean, and should you
believe it? It decides what biology can be read from the map: at ~8 Å you see α-helices as rods; at ~4 Å you trace the
protein backbone; at ~3 Å you see side chains and can discuss how a drug binds. A wrong resolution claim means wrong
biology — modelling side chains into noise, or "seeing" a ligand that is not there.

Resolution is never read off a picture by eye. It is **measured statistically**, by asking: if I split my data into
two independent halves, up to which level of detail do the two halves agree? This module builds that measurement
(the **Fourier shell correlation**), shows how it can be fooled, extends it to **local** resolution (different parts of
a molecule are resolved differently), and finally uses it as an objective to **choose which particles to keep**.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "(not needed here)")
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")
import os
print("CPU cores visible:", os.cpu_count())

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import multiprocessing as mp
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the helpers this notebook uses from [`course_toolkit.py`](../course_toolkit.py). The stars today
are `fsc` (Fourier shell/ring correlation, 3D or 2D) and `resolution_at` (where an FSC curve crosses a threshold), plus
`fetch_ngl_map` to download real cryo-EM maps.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

NGL_COMMIT = "0db88033650614103be761f151efc923b66e977e"
NGL_URL = "https://raw.githubusercontent.com/nglviewer/ngl/" + NGL_COMMIT + "/data/{name}"
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")


def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def fetch_ngl_map(name: str = "betaGal.mrc") -> tuple[np.ndarray, float, str] | None:
    """Download a real single-particle cryo-EM map distributed with the NGL viewer."""
    dest = os.path.join(CACHE_DIR, "ngl_" + name)
    if _download(NGL_URL.format(name=name), dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"NGL test map {name}"
    return None


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


print("🧰 toolkit loaded: fetch_ngl_map, fsc, resolution_at, lowpass, rotate_image, …")

### Data: a real cryo-EM map of β-galactosidase

**β-galactosidase** is the *E. coli* enzyme that splits lactose (the *lacZ* gene product of the famous *lac* operon) —
a ~465 kDa homotetramer and a classic test specimen for single-particle cryo-EM. The NGL viewer's test data include a
map of it (100³ voxels at 3.54 Å/voxel) and a matching **local-resolution map** (Section 6).

In [ ]:
# ── Load β-galactosidase (with an offline fallback) ──
def load_betagal():
    """Return (map, voxel size Å, source, local-resolution map or None)."""
    got = fetch_ngl_map("betaGal.mrc")
    if got is not None:
        vol, px, src = got
        lr = fetch_ngl_map("localResolution.mrc")
        return vol, px, src, (lr[0] if lr is not None else None)
    g = np.arange(100) - 49.5                                   # offline stand-in: a D2 arrangement of 4 lobes
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    vol = sum(np.exp(-((z - a) ** 2 / 90 + (y - b) ** 2 / 160 + (x - c) ** 2 / 60))
              for a, b, c in [(10, 14, 9), (10, -14, -9), (-10, 14, -9), (-10, -14, 9)])
    vol = vol * (1 + 0.3 * np.cos(z / 2.0) * np.cos(x / 2.5))       # some internal texture
    return vol.astype(np.float32), 3.54, "procedural stand-in (offline fallback)", None


betagal, PX, SOURCE, LOCALRES_NGL = load_betagal()
BOX = betagal.shape[0]
print(f"{SOURCE}: {betagal.shape} voxels at {PX} Å/voxel → box {BOX * PX:.0f} Å; "
      f"Nyquist resolution 2 × {PX} = {2 * PX:.2f} Å")
print("local-resolution map:", "available" if LOCALRES_NGL is not None else "not available")

---
> 📍 **Where we are.** Module 03 introduced spatial frequency and Fourier shells; Module 04 showed that averaging $N$
> noisy copies reduces noise by $\sqrt N$; Module 06 explained why every cryo-EM image is so noisy. Here we combine them
> into the field's central quality measure. Later modules rely on it constantly: subtomogram averages (Module 13) are
> judged by FSC, Multi-Pattern Pursuit (Module 14) ranks candidate patterns by an FSC-like score, and the capstone
> (Module 26) reports half-map FSC for every discovered pattern.

## 1 · Resolution is a frequency

Recall from Module 03: an image is a sum of waves. A wave with **spatial frequency** $k$ (cycles per Å) repeats every
$d=1/k$ Å. Saying a map has "resolution $d$" means: waves up to frequency $1/d$ are reliably present, finer ones are not.
On a grid with pixel size $p$, the finest representable wave has period $2p$ (**Nyquist**): $k_{\max}=1/(2p)$.

In 3D, all Fourier coefficients with the same $|k|$ form a **shell**; shell $i$ of an $N$-voxel box sits at
$|k|\approx i/(Np)$, i.e. at resolution $d\approx Np/i$.

In [ ]:
# ── 1.1 What different resolutions look like ──
z_mid = BOX // 2
CROP = (slice(BOX // 2 - 32, BOX // 2 + 32),) * 2                 # show the central 64 × 64 voxels (227 Å)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
axes[0].imshow(betagal[z_mid][CROP]); axes[0].set_title(f"original ({2 * PX:.1f} Å Nyquist)", fontsize=9)
for ax, d in zip(axes[1:], [10, 15, 25, 40]):
    ax.imshow(lowpass(betagal, cutoff=PX / d, soft=0.01)[z_mid][CROP]); ax.set_title(f"low-passed to {d} Å", fontsize=9)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
for i in (5, 10, 25, 50):
    print(f"shell {i:2d} of a {BOX}-voxel box at {PX} Å/voxel ↔ |k| = {i / (BOX * PX):.4f} Å⁻¹ ↔ d = {BOX * PX / i:5.1f} Å")

🔍 **What just happened?** Removing high frequencies removes *detail*, not intensity: at 10 Å the tetramer's domains are
still crisp, at 25 Å they merge into blobs, at 40 Å only the overall shape survives. A "resolution" is therefore a
statement about **which Fourier shells carry trustworthy signal**.

---
## 2 · Half-maps and the Fourier shell correlation

**The gold-standard recipe.** Split the particle images randomly into two halves *before* any refinement; process each
half completely independently; you get two **half-maps** $V_1$, $V_2$. Both contain the same signal but *independent*
noise. Compare them shell by shell:

$$\mathrm{FSC}(k)=\frac{\sum_{\mathbf k\in\text{shell}(k)}F_1(\mathbf k)\,F_2^{*}(\mathbf k)}{\sqrt{\sum_{\mathbf k\in\text{shell}(k)}|F_1(\mathbf k)|^2\;\sum_{\mathbf k\in\text{shell}(k)}|F_2(\mathbf k)|^2}},$$

*in words:* within each shell, compute the correlation coefficient between the two half-maps' Fourier coefficients
($^*$ = complex conjugate; the real part of the numerator is used). Where the signal dominates, FSC ≈ 1; where noise
dominates, FSC ≈ 0.

**Simulating half-maps.** Averaging $N/2$ particle images, each with noise of standard deviation $\sigma_p$, leaves noise
with standard deviation $\sigma_p/\sqrt{N/2}$ (Module 04). So a half-map from $N/2$ particles can be simulated in one
step as *truth + Gaussian noise of that size*. We also blur the truth with a **B-factor** envelope $e^{-Bk^2/4}$
($B=100$ Å²) that stands in for everything that dampens high-resolution signal in real data (alignment errors, radiation
damage, beam-induced motion).

In [ ]:
# ── 2.1 Simulate two half-maps from N particles ──
K3 = radial_frequency(betagal.shape) / PX                          # |k| in 1/Å for every 3D Fourier coefficient
B_FACTOR = 100.0                                                    # Å²
truth = np.real(np.fft.ifftn(np.fft.fftn(betagal) * np.exp(-B_FACTOR * K3 ** 2 / 4))).astype(np.float32)
SIGMA_PARTICLE = 40 * truth.std()                                  # noise of ONE particle (per-voxel SNR ≈ 6 × 10⁻⁴)


def half_maps(n_particles: int, rng) -> tuple[np.ndarray, np.ndarray]:
    """Two independent half-maps, each the average of n_particles/2 noisy copies of the (B-blurred) truth."""
    s = SIGMA_PARTICLE / np.sqrt(n_particles / 2)
    return (truth + rng.normal(0, s, truth.shape)).astype(np.float32), \
           (truth + rng.normal(0, s, truth.shape)).astype(np.float32)


N_PART = 4000
h1, h2 = half_maps(N_PART, np.random.default_rng(1))
# a soft mask around the molecule (we explain why in Section 3)
smooth = ndimage.gaussian_filter(betagal, 2.0)
core = smooth > 0.1 * smooth.max()
soft_mask = ndimage.gaussian_filter(ndimage.binary_dilation(core, iterations=3).astype(np.float32), 2.0)
freq, fsc_half, n_coef = fsc(h1 * soft_mask, h2 * soft_mask, PX)
d143, d05 = resolution_at(freq, fsc_half, 0.143), resolution_at(freq, fsc_half, 0.5)
print(f"N = {N_PART} particles, soft mask: FSC = 0.143 at {d143:.2f} Å, FSC = 0.5 at {d05:.2f} Å")
print("coefficients per shell (first 6):", n_coef[:6], "… → low shells are noisy estimates")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.1), gridspec_kw=dict(width_ratios=[1, 1, 1.6]))
axes[0].imshow(h1[z_mid][CROP]); axes[0].set_title("half-map 1 (slice)"); axes[0].axis("off")
axes[1].imshow(h2[z_mid][CROP]); axes[1].set_title("half-map 2 (slice)"); axes[1].axis("off")
axes[2].plot(freq, fsc_half, color="tab:blue")
for thr, col in [(0.143, "tab:red"), (0.5, "tab:orange")]:
    axes[2].axhline(thr, color=col, ls="--", lw=1, label=f"FSC = {thr}")
axes[2].axvline(1 / d143, color="tab:red", lw=0.8)
axes[2].set_xlabel("spatial frequency (1/Å)"); axes[2].set_ylabel("FSC"); axes[2].legend(fontsize=7)
axes[2].set_title(f"half-map FSC → {d143:.1f} Å at 0.143")
sec = axes[2].secondary_xaxis("top", functions=(lambda f: 1 / np.maximum(f, 1e-3), lambda d: 1 / np.maximum(d, 1e-3)))
sec.set_xticks([40, 20, 12, 9, 7.08]); sec.set_xlabel("resolution (Å)", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The FSC starts near 1 (low frequencies: overall shape, identical in both halves) and falls to
~0 where noise wins. Where to draw the line?

**Thresholds — be precise.**

| Threshold | Compare | Meaning |
|---|---|---|
| **0.143** | two independent **half-maps** (gold standard) | the *full* map (both halves averaged) then correlates with the (unknown) true structure at 0.5 |
| **0.5** | a map with a **known truth** (simulations), or a map with an atomic model | the map–truth correlation is 0.5 |

Why are they consistent? If each half-map has spectral signal-to-noise ratio $\text{SSNR}_{h}$ in a shell, then
$\text{FSC}_{\text{half}}=\text{SSNR}_h/(1+\text{SSNR}_h)$ (Section 4), and the full map, with twice the SSNR, correlates
with the truth as

$$C_{\text{ref}}=\sqrt{\frac{2\,\text{FSC}_{\text{half}}}{1+\text{FSC}_{\text{half}}}}\qquad(\text{Rosenthal \& Henderson 2003}).$$

*In words:* the correlation of the final map with the truth can be predicted from the half-map FSC; plugging in
$\text{FSC}_{\text{half}}=0.143$ gives $C_{\text{ref}}=\sqrt{0.286/1.143}=0.5$. In a simulation we *know* the truth, so we
can check this.

In [ ]:
# ── 2.2 Check: half-map FSC at 0.143 ↔ full-map-vs-truth FSC at 0.5 ──
full = (h1 + h2) / 2
_, fsc_truth, _ = fsc(full * soft_mask, truth * soft_mask, PX)
c_ref_pred = np.sqrt(2 * np.clip(fsc_half, 0, 1) / (1 + np.clip(fsc_half, 0, 1)))
print(f"half-maps at 0.143: {d143:.2f} Å   |   full map vs truth at 0.5: {resolution_at(freq, fsc_truth, 0.5):.2f} Å")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(freq, fsc_half, label="FSC(half 1, half 2)")
ax.plot(freq, fsc_truth, label="FSC(full map, truth) — measured")
ax.plot(freq, c_ref_pred, "k:", label="√(2·FSC_half / (1 + FSC_half)) — predicted")
ax.axhline(0.143, color="tab:red", ls="--", lw=0.8); ax.axhline(0.5, color="tab:orange", ls="--", lw=0.8)
ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("correlation"); ax.legend(fontsize=7)
ax.set_title("two thresholds, one resolution")
plt.tight_layout(); plt.show()
assert abs(d143 - resolution_at(freq, fsc_truth, 0.5)) < 1.0, "the two criteria should agree to within ~1 Å"

🔍 **What just happened?** The predicted curve (dotted) lies on the measured map-vs-truth curve (beyond the 0.143 point
it wobbles, because the square root magnifies the noise of FSC values near zero), and the two resolution criteria
agree. So: **0.143 for half-maps, 0.5 when comparing with a known truth or model.** Using 0.5 on half-maps is
merely conservative; using 0.143 on a map-vs-truth comparison is wrong (it would overstate resolution). Other criteria
exist (e.g. the ½-bit criterion of van Heel & Schatz 2005); always report which one you used.

---
## 3 · How FSC can lie

### 3.1 Masks

97 % of our box is solvent that contains only noise. Comparing whole boxes dilutes the signal, so FSC is usually
computed inside a **mask** around the molecule. But a mask multiplies *both* half-maps by the same function: a mask with
**sharp edges** adds the same sharp-edged pattern to both, which correlates at high frequency even though no extra
signal exists. Remedies: use **soft** masks (smooth edges), and check with **phase randomisation** (Chen *et al.* 2013):
randomise the Fourier phases of both half-maps beyond some frequency (in practice where the unmasked FSC is still
≈ 0.8), destroying true signal there; apply the same mask, see how much correlation the mask alone produces; then
correct

$$\text{FSC}_{\text{true}}=\frac{\text{FSC}_{\text{masked}}-\text{FSC}_{\text{rand}}}{1-\text{FSC}_{\text{rand}}}.$$

*In words:* whatever correlation survives after the signal has been scrambled is caused by the mask, and is subtracted.

In [ ]:
# ── 3.1 No mask, soft mask, sharp tight mask, and the phase-randomisation correction ──
tight_mask = (ndimage.gaussian_filter(truth, 1.0) > 0.25 * truth.max()).astype(np.float32)    # binary, hugs density


def phase_randomize(vol: np.ndarray, beyond_A: float, rng) -> np.ndarray:
    """Replace the Fourier phases of all coefficients finer than ``beyond_A`` by random phases (amplitudes kept)."""
    F = np.fft.fftn(vol)
    sel = K3 > 1 / beyond_A
    F[sel] = np.abs(F[sel]) * np.exp(1j * rng.uniform(0, 2 * np.pi, sel.sum()))
    return np.real(np.fft.ifftn(F)).astype(np.float32)


curves = {}
for name, m in [("no mask", None), ("soft mask", soft_mask), ("sharp tight mask", tight_mask)]:
    a, b = (h1, h2) if m is None else (h1 * m, h2 * m)
    curves[name] = fsc(a, b, PX)[1]
RAND_FROM = resolution_at(freq, curves["no mask"], 0.8)           # Å: randomise beyond where unmasked FSC < 0.8
r1 = phase_randomize(h1, RAND_FROM, np.random.default_rng(11))
r2 = phase_randomize(h2, RAND_FROM, np.random.default_rng(12))
fsc_rand = fsc(r1 * tight_mask, r2 * tight_mask, PX)[1]
corrected = np.where(freq > 1 / RAND_FROM,
                     (curves["sharp tight mask"] - fsc_rand) / (1 - fsc_rand), curves["sharp tight mask"])
curves["tight mask, corrected"] = corrected
fig, ax = plt.subplots(figsize=(6.6, 3.4))
for (name, c), ls in zip(curves.items(), ["-", "-", "-", "--"]):
    ax.plot(freq, c, ls, label=f"{name}: {resolution_at(freq, c, 0.143):.1f} Å")
ax.plot(freq, fsc_rand, ":", color="gray", label="tight mask on phase-randomised maps")
ax.axhline(0.143, color="tab:red", lw=0.8, ls="--"); ax.axvline(1 / RAND_FROM, color="gray", lw=0.6)
ax.set_ylim(-0.3, 1.05); ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("FSC"); ax.legend(fontsize=7)
ax.set_title("the same half-maps, four answers")
plt.tight_layout(); plt.show()
for name, c in curves.items():
    print(f"{name:24s} → {resolution_at(freq, c, 0.143):5.2f} Å")
print(f"(phases randomised beyond {RAND_FROM:.1f} Å, where the unmasked FSC falls below 0.8)")
print(f"(map vs truth at 0.5 says {resolution_at(freq, fsc_truth, 0.5):.2f} Å)")

🔍 **What just happened?** One pair of half-maps, four "resolutions". Without a mask the solvent noise drags the FSC
down (too pessimistic). The soft mask gives the honest answer (it matches the map-vs-truth value). The sharp tight mask
keeps the FSC high almost to Nyquist — but the phase-randomised maps (gray dotted) show that most of that correlation is
produced by the mask itself; after the correction the claim drops back (here even a little below the soft-mask value,
because our tight mask also cuts into the molecule; the wiggle just above the randomisation frequency comes from the
mask smearing un-randomised low-frequency signal). ⚠️ When a paper's FSC curve stays high right up to Nyquist, ask
about the mask.

### 3.2 Independence: "Einstein from noise"

FSC assumes the two halves have **independent** noise. Refinement can break this. A famous demonstration (Henderson
2013, "Einstein from noise"): take images of **pure noise**, align each to a reference picture by searching rotations
and shifts for the best cross-correlation, and average. The average looks like the reference! The alignment has selected,
from each noise image, the pattern that resembles the reference. If both half-sets are aligned to the *same* reference,
both averages contain the same reference-shaped "signal" and their FSC is high — for data with **no signal at all**.

Gold-standard refinement prevents this: each half is refined against its *own* reference, and the starting reference is
**low-pass filtered** (e.g. to 40–60 Å) so that no high-resolution detail can be imprinted on both halves.

In [ ]:
# ── 3.2 Align pure noise to a reference and measure the FRC between the half-set averages ──
PX2 = 2 * PX                                                        # 2D images binned 2× → 7.08 Å/pixel
proj = betagal.sum(axis=0)
ref2d = proj.reshape(BOX // 2, 2, BOX // 2, 2).mean(axis=(1, 3))
ref2d = (ref2d - ref2d.mean()) / ref2d.std()
S2 = ref2d.shape[0]
K2 = radial_frequency((S2, S2)) / PX2


def align_to_reference(img: np.ndarray, ref_fts: list) -> np.ndarray:
    """Find the rotation (from a list) and shift maximising cross-correlation with a reference; return the aligned image."""
    best_cc, best = -np.inf, None
    Fi = np.fft.fft2(img)
    for ang, Rf in ref_fts:
        cc = np.real(np.fft.ifft2(Fi * np.conj(Rf)))
        i = int(np.argmax(cc))
        if cc.flat[i] > best_cc:
            best_cc, best = cc.flat[i], (ang, np.unravel_index(i, cc.shape))
    ang, (dy, dx) = best
    return rotate_image(np.roll(img, (-dy, -dx), axis=(0, 1)), -ang)


def noise_alignment_experiment(reference: np.ndarray, n_img: int, rng, step_deg: float = 30):
    """Align n_img pure-noise images to copies of ``reference`` rotated every ``step_deg``°, searching all shifts;
    return (average, FRC of the two half averages)."""
    ref_fts = [(a, np.fft.fft2(rotate_image(reference, a))) for a in np.radians(np.arange(0, 360, step_deg))]
    aligned = np.stack([align_to_reference(rng.normal(0, 1, (S2, S2)), ref_fts) for _ in range(n_img)])
    return aligned.mean(0), fsc(aligned[0::2].mean(0), aligned[1::2].mean(0), PX2)


t0 = time.time()
ref_lp = np.real(np.fft.ifft2(np.fft.fft2(ref2d) * (K2 <= 1 / 40.0)))      # reference low-passed to 40 Å
avg_full, (f2, frc_full, _) = noise_alignment_experiment(ref2d, 400, np.random.default_rng(21))
avg_lp, (_, frc_lp, _) = noise_alignment_experiment(ref_lp, 400, np.random.default_rng(22))
rngn = np.random.default_rng(23)
frc_indep = fsc(rngn.normal(0, 1, (200, S2, S2)).mean(0), rngn.normal(0, 1, (200, S2, S2)).mean(0), PX2)[1]
print(f"{time.time() - t0:.1f} s | corr(average of aligned NOISE, reference) = {pearson(avg_full, ref2d):.2f}")
print(f"FRC 0.143 'resolution' from pure noise: shared full-resolution reference {resolution_at(f2, frc_full):.1f} Å, "
      f"shared 40 Å reference {resolution_at(f2, frc_lp):.1f} Å")

fig, axes = plt.subplots(1, 4, figsize=(11.5, 2.9), gridspec_kw=dict(width_ratios=[1, 1, 1, 1.7]))
for ax, im, t in zip(axes[:3], [ref2d, rng.normal(0, 1, (S2, S2)), avg_full],
                     ["reference (β-gal projection)", "one 'particle' (pure noise)", "average of 400 aligned noise images"]):
    ax.imshow(im); ax.set_title(t, fontsize=8); ax.axis("off")
axes[3].plot(f2[1:], frc_full[1:], label="aligned to shared full-res reference")
axes[3].plot(f2[1:], frc_lp[1:], label="aligned to shared 40 Å reference")
axes[3].plot(f2[1:], frc_indep[1:], label="not aligned (independent noise)")
axes[3].axhline(0.143, color="tab:red", ls="--", lw=0.8); axes[3].axvline(1 / 40, color="gray", lw=0.6)
axes[3].set_xlabel("spatial frequency (1/Å)"); axes[3].set_ylabel("FRC of half averages"); axes[3].legend(fontsize=6.5)
axes[3].set_title("FRC from data with NO signal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Four hundred images of pure noise, once aligned to β-galactosidase, average to a recognisable
β-galactosidase (correlation printed above), and their half-set FRC claims a "resolution" — a textbook case of
**reference bias** (Module 13 returns to it). With a 40 Å low-passed reference, the fake agreement is confined to
frequencies below 1/40 Å⁻¹; beyond, the halves are independent again and the FRC drops to ~0, as it should for noise. This
is why gold-standard refinement low-pass filters the initial reference and never lets the two halves see each other.

✏️ **Your turn.** In 3.2, call `noise_alignment_experiment(ref2d, 400, np.random.default_rng(21), step_deg=90)` (only 4
rotations) or use 40 images instead of 400. Does the illusion get weaker or stronger? Why does a *larger search* make
overfitting worse?

---
## 4 · Spectral SNR and FSC: the formulas behind MPP and Shi *et al.*

Suppose $n$ aligned images share a signal $S(\mathbf k)$ plus independent noise of power $\sigma^2(\mathbf k)$. The
**spectral signal-to-noise ratio** of their average, in a shell, is

$$\eta=\frac{n\,\sum_{\text{shell}}|S|^2}{\sum_{\text{shell}}\sigma^2},$$

*in words:* the signal power over the noise power of the *average* (averaging $n$ images divides the noise power by $n$).
It can be **estimated from the data** (Unser *et al.* 1987; Penczek 2002) from the mean $\bar a$ and the variance
$\hat\sigma^2$ of the Fourier coefficients across images:

$$\hat\eta=\frac{\sum_{\text{shell}} n\,|\bar a|^2}{\sum_{\text{shell}}\hat\sigma^2}-1 .$$

The "−1" removes a bias: even pure noise has $n|\bar a|^2\approx\sigma^2$. (⚠️ The formula printed in Shi *et al.* 2020
and implemented in AITom's `ssnr2d.py` and MPP's `ssnr.py` computes the ratio *without* the −1, so pure noise scores
$\hat\eta\approx1$ rather than 0.)

**From SSNR to FSC.** Split the $n$ images into two halves; each half-average has SSNR $\eta_h=\eta/2$. For two
half-averages $F_1=S+N_1$, $F_2=S+N_2$ with independent noises, the expected FSC is

$$\text{FSC}_{\text{half}}=\frac{|S|^2}{|S|^2+\sigma^2_h}=\frac{\eta_h}{1+\eta_h}=\frac{\eta/2}{1+\eta/2}=\frac{\eta}{2+\eta}.$$

*In words:* knowing the SSNR of the **full** set, $\rho=\eta/(2+\eta)$ predicts the FSC you *would* measure between two
random half-sets — without having to split. Multi-Pattern Pursuit (Xu *et al.* 2019) uses exactly
$\rho_r=\eta_r/(2+\eta_r)$ and scores a candidate average by $\text{SFSC}=\sum_r\rho_r$; Shi *et al.* 2020 use
$c(J)=s(J)/(2+s(J))$ with $r(S)=\sum_J c(J)$. Let us verify the relation.

In [ ]:
# ── 4.1 SSNR from n aligned noisy images vs FRC between random half-sets ──
N_SH2 = S2 // 2
IDX2 = np.minimum((radial_frequency((S2, S2)) * 2 * N_SH2).astype(int), N_SH2).ravel()


def ssnr_rings(sum_F: np.ndarray, sum_P: np.ndarray, n: int, bias_correct: bool = True) -> np.ndarray:
    """Ring-wise SSNR of the average of n images from running sums of Fourier coefficients (sum_F) and powers (sum_P)."""
    mean2 = np.abs(sum_F.ravel()) ** 2 / n ** 2
    var = (sum_P.ravel() - n * mean2) / (n - 1)                     # unbiased variance across images
    num = np.bincount(IDX2, n * mean2, N_SH2 + 1)[:N_SH2]
    den = np.bincount(IDX2, var, N_SH2 + 1)[:N_SH2]
    eta = num / np.maximum(den, 1e-12)
    return np.maximum(eta - 1, 0) if bias_correct else eta


n_img, snr_img = 400, 0.05
imgs = ref2d[None] + np.random.default_rng(31).normal(0, np.sqrt(ref2d.var() / snr_img), (n_img, S2, S2))
F_imgs = np.fft.fft2(imgs)
eta = ssnr_rings(F_imgs.sum(0), (np.abs(F_imgs) ** 2).sum(0), n_img, bias_correct=True)
eta_raw = ssnr_rings(F_imgs.sum(0), (np.abs(F_imgs) ** 2).sum(0), n_img, bias_correct=False)
perm = np.random.default_rng(32).permutation(n_img)
f2, frc_split, _ = fsc(imgs[perm[:n_img // 2]].mean(0), imgs[perm[n_img // 2:]].mean(0), PX2)
fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.plot(f2[1:], frc_split[1:], "o-", ms=3, label="measured FRC of two random half-sets")
ax.plot(f2[1:], (eta / (2 + eta))[1:], "-", lw=2, alpha=0.7, label="ρ = η/(2+η), bias-corrected η")
ax.plot(f2[1:], (eta_raw / (2 + eta_raw))[1:], "--", label="same, η without the −1 (paper/AITom form)")
ax.axhline(1 / 3, color="gray", lw=0.6, ls=":")
ax.set_xlabel("spatial frequency (1/Å)"); ax.set_ylabel("correlation"); ax.legend(fontsize=7)
ax.set_title(f"{n_img} images at SNR {snr_img}: FSC predicted from SSNR")
plt.tight_layout(); plt.show()
hi = f2 > 1 / 20
print(f"mean |measured − predicted| beyond 20 Å: bias-corrected {np.abs(frc_split - eta / (2 + eta))[hi].mean():.3f}, "
      f"uncorrected {np.abs(frc_split - eta_raw / (2 + eta_raw))[hi].mean():.3f}")

🔍 **What just happened?** The FRC predicted from the full set's SSNR (thick line) follows the FRC actually measured by
splitting the data — the formula works. Without the "−1" the prediction never drops below ~1/3 (gray dotted line), because
pure-noise shells still score $\hat\eta\approx1$. For *ranking* subsets of the same data this offset matters little (both
versions increase with the true SSNR), which is how Shi *et al.* and MPP use it; for *reporting* an FSC it matters.

### 📄 Paper connection (preview) — the SFSC score of Multi-Pattern Pursuit

**Paper**: Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F. *De Novo Structural Pattern Mining in
Cellular Electron Cryotomograms.* Structure 27(4):679–691 (2019). [doi:10.1016/j.str.2019.01.005](https://doi.org/10.1016/j.str.2019.01.005)

- **Relevant piece:** every candidate pattern (an average of aligned subtomograms) is scored by
  $\text{SFSC}=\sum_r\rho_r$, with $\rho_r=\eta_r/(2+\eta_r)$ and $\eta_r$ the SSNR of the aligned set, computed with
  missing-wedge counts $\tilde M(\xi)$ (how many subtomograms observe each Fourier coefficient):
  $\eta_r=\int_{||\xi|-r|\le\Delta r}\tilde M(\xi)|\mu(\xi)|^2\big/\int_{||\xi|-r|\le\Delta r}\sigma^2(\xi)$.
  The code (`ssnr_to_fsc(ssnr) = ssnr/(2+ssnr)`) matches.
- **Why it is clever:** the expected half-set FSC is computed from *all* members at once, so every candidate pattern
  can be scored quickly without splitting. Module 14 builds the full MPP loop around it.

---
## 5 · More particles, better maps — but slowly

Each doubling of the number of particles halves the noise power, i.e. adds a constant to the SSNR in *log* units. Since
signal power falls off at high frequency roughly like $e^{-Bk^2/2}$, the resolution $d$ reached with $N$ particles obeys
approximately

$$\ln N \approx \text{const}+\frac{B_{\text{overall}}}{2}\cdot\frac{1}{d^2}\qquad(\text{the Rosenthal–Henderson plot}),$$

*in words:* plotting $1/d^2$ against $\ln N$ gives a straight line whose slope is $2/B_{\text{overall}}$; a large
overall B-factor means each extra ångström costs many more particles.

In [ ]:
# ── 5.1 FSC and resolution versus the number of particles ──
N_LIST = [250, 1000, 4000, 16000, 64000]
res_N = []
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
for N in N_LIST:
    a, b = half_maps(N, np.random.default_rng(N))
    f, c, _ = fsc(a * soft_mask, b * soft_mask, PX)
    res_N.append(resolution_at(f, c, 0.143))
    axes[0].plot(f, c, label=f"N = {N:,}: {res_N[-1]:.1f} Å")
axes[0].axhline(0.143, color="tab:red", ls="--", lw=0.8)
axes[0].set_xlabel("spatial frequency (1/Å)"); axes[0].set_ylabel("half-map FSC"); axes[0].legend(fontsize=7)
axes[0].set_title("more particles → FSC stays high longer")
inv_d2 = 1 / np.array(res_N) ** 2
slope, icpt = np.polyfit(np.log(N_LIST), inv_d2, 1)
axes[1].plot(np.log(N_LIST), inv_d2, "o", color="tab:purple")
axes[1].plot(np.log(N_LIST), slope * np.log(N_LIST) + icpt, "-", color="tab:purple", lw=1)
for N, d in zip(N_LIST, res_N):
    axes[1].annotate(f"{d:.1f} Å", (np.log(N), 1 / d ** 2), textcoords="offset points", xytext=(4, -10), fontsize=7)
axes[1].set_xlabel("ln N (number of particles)"); axes[1].set_ylabel("1 / d²  (Å⁻²)")
axes[1].set_title(f"Rosenthal–Henderson plot: B_overall ≈ {2 / slope:.0f} Å²")
plt.tight_layout(); plt.show()
print("resolution (Å):", dict(zip(N_LIST, np.round(res_N, 2))))
print(f"fitted overall B-factor ≈ {2 / slope:.0f} Å² (we imposed B = {B_FACTOR:.0f} Å² on top of the map's own fall-off)")

🔍 **What just happened?** Resolution improves ever more slowly: a 256-fold increase in particles buys only a few
ångströms, and the points fall close to a straight line in the Rosenthal–Henderson plot (the last point also feels the
7.08 Å Nyquist limit). The fitted $B_{\text{overall}}$ is much larger than the
100 Å² we imposed, because the β-galactosidase map itself already has weak high-resolution power (it was filtered and
sampled at 3.54 Å) — $B_{\text{overall}}$ includes *every* source of high-frequency loss. In real projects this plot
tells you whether collecting more data is worth it.

---
## 6 · Local resolution: not every part of a molecule is equally sharp

🧫 Flexible loops, mobile domains and partially occupied subunits are blurred in the average, while the rigid core is
sharp. A single global number hides this. **Local resolution** repeats the FSC in a small window around every position:
slide a cubic window (with soft edges) over both half-maps, compute the FSC inside, and assign the 0.143-crossing to the
window's centre (blocres and Lü *et al.* 2020 compute windowed FSCs like this; ResMap instead uses a local statistical
test).

**Our experiment.** We make half-maps in which the blur increases from the centre of β-galactosidase outward
($B$ = 50 Å² in the core to 600 Å² at the periphery — like a rigid core with floppy surface regions) and check whether the
sliding-window FSC recovers this pattern.

In [ ]:
# ── 6.1 Half-maps with a spatially varying blur ──
B_LEVELS = [50, 150, 300, 600]                                      # Å², core → periphery
F_bg = np.fft.fftn(betagal)
blurred = np.stack([np.real(np.fft.ifftn(F_bg * np.exp(-B * K3 ** 2 / 4))) for B in B_LEVELS]).astype(np.float32)
grid = np.meshgrid(*[np.arange(BOX) - (BOX - 1) / 2] * 3, indexing="ij")
radius = np.sqrt(sum(g ** 2 for g in grid))
r_max = radius[core].max()
level = np.clip((radius - 5) / (r_max - 5), 0, 1) * (len(B_LEVELS) - 1)         # 0 … 3, grows with radius
lo_i = np.floor(level).astype(int); hi_i = np.minimum(lo_i + 1, len(B_LEVELS) - 1); w = level - lo_i
truth_local = ((1 - w) * np.take_along_axis(blurred, lo_i[None], 0)[0]
               + w * np.take_along_axis(blurred, hi_i[None], 0)[0]).astype(np.float32)
B_design = np.interp(level, np.arange(len(B_LEVELS)), B_LEVELS)          # the blur we imposed, per voxel
noise_sd = 0.6 * truth_local[core].std()
rng6 = np.random.default_rng(61)
lh1 = (truth_local + rng6.normal(0, noise_sd, truth_local.shape)).astype(np.float32)
lh2 = (truth_local + rng6.normal(0, noise_sd, truth_local.shape)).astype(np.float32)
fg, cg, _ = fsc(lh1 * soft_mask, lh2 * soft_mask, PX)
print(f"global (soft-masked) resolution of these half-maps: {resolution_at(fg, cg):.2f} Å — one number for the whole molecule")

In [ ]:
# ── 6.2 Sliding-window FSC ──
WIN, STEP = 20, 4                                                    # window 20 voxels = 71 Å; step 4 voxels
hann = np.hanning(WIN + 2)[1:-1]
TAPER = (hann[:, None, None] * hann[None, :, None] * hann[None, None, :]).astype(np.float32)   # soft-edged window
N_SHW = WIN // 2
IDXW = np.minimum((radial_frequency((WIN,) * 3) * 2 * N_SHW).astype(int), N_SHW).ravel()
FREQW = (np.arange(N_SHW) + 0.5) / (2 * N_SHW) / PX


def window_resolution(a: np.ndarray, b: np.ndarray, start) -> float:
    """FSC between two half-maps inside one tapered window starting at ``start`` (z, y, x) → resolution at 0.143 (Å)."""
    z, y, x = start
    A = np.fft.fftn(a[z:z + WIN, y:y + WIN, x:x + WIN] * TAPER)
    Bw = np.fft.fftn(b[z:z + WIN, y:y + WIN, x:x + WIN] * TAPER)
    num = np.bincount(IDXW, (A * np.conj(Bw)).real.ravel(), N_SHW + 1)[:N_SHW]
    d1 = np.bincount(IDXW, (np.abs(A) ** 2).ravel(), N_SHW + 1)[:N_SHW]
    d2 = np.bincount(IDXW, (np.abs(Bw) ** 2).ravel(), N_SHW + 1)[:N_SHW]
    return resolution_at(FREQW, num / np.sqrt(d1 * d2 + 1e-20), 0.143)


# only windows centred inside the molecule (the 'mask' of local-resolution programs)
starts = [(z, y, x) for z in range(0, BOX - WIN + 1, STEP) for y in range(0, BOX - WIN + 1, STEP)
          for x in range(0, BOX - WIN + 1, STEP) if core[z + WIN // 2, y + WIN // 2, x + WIN // 2]]
t0 = time.time()
local_res = np.array([window_resolution(lh1, lh2, s) for s in starts])
T_SERIAL = time.time() - t0
centres = np.array(starts) + WIN // 2
print(f"{len(starts)} windows in {T_SERIAL:.1f} s; local resolution {local_res.min():.1f}–{local_res.max():.1f} Å")
rho = np.corrcoef(local_res, B_design[tuple(centres.T)])[0, 1]
print(f"correlation between measured local resolution and the imposed blur B: {rho:.2f}")
assert rho > 0.4, "local resolution should track the imposed blur"

In [ ]:
# ── 6.3 From window centres to a full map (cubic interpolation, as in Lü et al.) ──
n_pos = len(range(0, BOX - WIN + 1, STEP))
coarse = np.full((n_pos,) * 3, np.nan)
coarse[tuple((np.array(starts) // STEP).T)] = local_res
nearest = ndimage.distance_transform_edt(np.isnan(coarse), return_distances=False, return_indices=True)
coarse_filled = coarse[tuple(nearest)]                              # fill outside-mask nodes with the nearest value
coords = (np.arange(BOX) - WIN // 2) / STEP                         # voxel → coarse-grid coordinate
zz, yy, xx = np.meshgrid(coords, coords, coords, indexing="ij")
local_map = ndimage.map_coordinates(coarse_filled, [zz, yy, xx], order=3, mode="nearest")

fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.1))
sl = BOX // 2
axes[0].imshow(truth_local[sl][CROP]); axes[0].set_title("designed half-map truth (slice)")
im1 = axes[1].imshow(np.where(core[sl], B_design[sl], np.nan)[CROP], cmap="viridis"); axes[1].set_title("imposed blur B (Å²)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(np.where(core[sl], local_map[sl], np.nan)[CROP], cmap="viridis_r")
axes[2].set_title("measured local resolution (Å)")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
for ax in axes[:3]:
    ax.set_xticks([]); ax.set_yticks([])
axes[3].scatter(B_design[tuple(centres.T)], local_res, s=6, alpha=0.5)
axes[3].set_xlabel("imposed B at window centre (Å²)"); axes[3].set_ylabel("local resolution (Å)")
axes[3].set_title(f"windows: r = {rho:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The local-resolution map (third panel; yellow = better) is sharpest in the core and degrades
outward, following the blur we imposed — information the single global number hid. The scatter is wide because each
71 Å window contains few Fourier coefficients per shell (noisy FSC estimates) and mixes regions of different blur:
window size is a trade-off between **statistical precision** (big windows) and **spatial detail** (small windows).

### 6.4 A real local-resolution map

The NGL test data ship a local-resolution map computed for this β-galactosidase map (we do not know which program or
parameters produced it, so we compare only qualitatively).

In [ ]:
# ── 6.4 The real local-resolution map of β-galactosidase ──
if LOCALRES_NGL is not None:
    lr_in = LOCALRES_NGL[core]
    print(f"real local resolution inside the molecule: {lr_in.min():.1f}–{lr_in.max():.1f} Å (median {np.median(lr_in):.1f} Å)")
    print(f"correlation with distance from the centre: {np.corrcoef(lr_in, radius[core])[0, 1]:.2f}")
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1))
    axes[0].imshow(betagal[sl][CROP]); axes[0].set_title("β-gal map (central slice)"); axes[0].axis("off")
    im = axes[1].imshow(np.where(core[sl], LOCALRES_NGL[sl], np.nan)[CROP], cmap="viridis_r")
    plt.colorbar(im, ax=axes[1], fraction=0.046, label="Å"); axes[1].set_title("NGL local resolution (slice)"); axes[1].axis("off")
    axes[2].hist(lr_in, bins=30, color="tab:green"); axes[2].set_xlabel("local resolution (Å)"); axes[2].set_ylabel("voxels")
    axes[2].set_title("inside the molecule")
    plt.tight_layout(); plt.show()
else:
    print("local-resolution map not available offline — skipping the comparison")

🔍 **What just happened?** The real map shows the same qualitative pattern as our toy: the best local resolution
(≈ 7 Å here, close to the 7.1 Å Nyquist limit of this 3.54 Å/voxel map) in the core, worse values toward the surface. A
biologist reading this map would trust side-chain-level interpretation only in the core.

---
## 7 · Parallel local FSC: the Spark idea, on your laptop

Local FSC is "embarrassingly parallel": every window is independent. Our toy has a few hundred windows; a real map
(e.g. 256³ voxels with 64³ windows and step 1) has *millions*, which took hours on one machine. Lü *et al.* (2020)
distributed the work with **Apache Spark**:

- Spark stores data as an **RDD** (resilient distributed dataset) split into **partitions**; a `map` function runs on
  every partition in parallel on many workers; if a worker dies, its partitions are recomputed elsewhere (fault tolerance).
- Their *basic* design put every voxel in the RDD as a key–value quadruple (x, y, z, value) — lots of data to shuffle.
- Their *optimised* design puts only the **window start coordinates** (keyed by sequence number) in the RDD; each
  partition cuts its own sub-volumes from the half-maps. Less shuffling, less memory.

We emulate the optimised design with Python's `concurrent.futures`: the "RDD" is our list of window starts, split into
partitions; worker processes share the half-maps and each maps `window_resolution` over its partition.

In [ ]:
# ── 7.1 'Spark-style' map over partitions of window coordinates ──
def process_partition(partition: list) -> list:
    """Worker task: compute the local resolution of every window start in one partition (half-maps are shared)."""
    return [window_resolution(lh1, lh2, s) for s in partition]


def parallel_local_fsc(starts: list, n_workers: int = 2, n_partitions: int = 4):
    """Split window starts into partitions, map them over a pool of workers, and reassemble in the original order."""
    partitions = [starts[i::n_partitions] for i in range(n_partitions)]
    try:                                                             # processes (true parallelism) where 'fork' exists
        ctx = mp.get_context("fork")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as ex:
                parts = list(ex.map(process_partition, partitions))
        backend = "processes"
    except (ValueError, OSError, RuntimeError):                      # e.g. Windows: fall back to threads
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            parts = list(ex.map(process_partition, partitions))
        backend = "threads"
    out = np.empty(len(starts))
    for i, p in enumerate(parts):
        out[i::n_partitions] = p
    return out, backend


print(f"basic design would ship {4 * BOX ** 3:,} numbers (x, y, z, value per voxel); "
      f"optimised design ships {3 * len(starts):,} (one start coordinate per window)")
t0 = time.time()
local_res_par, backend = parallel_local_fsc(starts, n_workers=2)
T_PAR = time.time() - t0
assert np.allclose(local_res_par, local_res), "parallel and serial results must be identical"
print(f"serial {T_SERIAL:.2f} s | parallel ({backend}, 2 workers) {T_PAR:.2f} s → speed-up ×{T_SERIAL / T_PAR:.2f}")
print("identical results: ✅")

🔍 **What just happened?** The parallel run gives *exactly* the same local resolutions — partitioning changes only who
computes what. The speed-up depends on your machine: on a free Colab instance (2 cores) expect up to about ×2 minus the
overhead of starting workers; on a busy shared computer or a 1-core machine you may see none. Spark's advantage appears at
scale: many machines, big maps, and automatic recovery from failures.

✏️ **Your turn.** Set `STEP = 2` in 6.2 (8× more windows) and re-run 6.2 and 7.1. Does the speed-up improve? (Bigger jobs
amortise the start-up overhead — Amdahl's law in action.)

### 📄 Paper connection — Lü *et al.* 2020: Spark-based parallel local FSC

**Paper**: Lü Y, Zeng X, Tian X, Shi X, Wang H, Zheng X, Liu X, Zhao X, Gao X, Xu M. *Spark-based parallel calculation of
3D Fourier shell correlation for macromolecule structure local resolution estimation.* BMC Bioinformatics 21(Suppl 13):391
(2020). [doi:10.1186/s12859-020-03680-6](https://doi.org/10.1186/s12859-020-03680-6) (listed by the lab as an APBC 2020
paper; published as a BMC Bioinformatics supplement article).

- **Question it answers:** local resolution by sliding-window FSC is slow and scales poorly on one machine — can a
  cluster make it fast?
- **What the paper actually does:** slides a cubic window (e.g. 64³) over two half-maps (optionally within a mask),
  computes the FSC (equation in Section 2) in each window, assigns the 0.143 crossing to the window centre and fills the
  remaining voxels by cubic interpolation; parallelises on Spark with (i) a basic RDD of (x, y, z, value) voxel
  quadruples and (ii) an optimised RDD of window start coordinates keyed by sequence number, where each partition cuts
  its own sub-arrays; relies on Spark's fault tolerance.
- **What our toy keeps:** tapered sliding windows, 0.143 crossing per window, cubic interpolation to a full map, the
  "ship coordinates, not voxels" partitioning, a map over partitions with identical results.
- **What our toy leaves out:** Spark itself (cluster, RDD shuffles, fault tolerance), 64³ windows on 256³ maps, the
  comparison with blocres/ResMap.
- **Result reported:** for human TFIIH (EMD-3802, 256³, 64³ windows) the authors report ≈ 106 min with blocres versus
  ≈ 25 min on 4 nodes and ≈ 13 min on 8 nodes (≈ 4–8× faster), with local-resolution histograms similar to blocres's
  (their accuracy comparison is qualitative).
- **Code:** <https://github.com/xulabs/projects>. Deep dive in this repo: [`../../LocalResolution/`](../../LocalResolution/).

---
## 8 · Choosing good particles: resolution-guided selection by simulated annealing

🧫 A particle stack always contains "bad" images: misaligned particles, damaged or partially unfolded molecules,
contaminants. Averaging them in blurs the map. If we knew which images were good, we would keep only those — but we
don't, and we don't even know how many there are. Shi *et al.* (2020) turn the resolution measure into an **objective**:
choose the subset $S$ of pre-aligned images whose average has the best resolution score

$$r(S)=\sum_J c(J),\qquad c(J)=\frac{s(J)}{2+s(J)},\qquad s(J)=\frac{\sum_{j\in J}\big|\sum_{i\in S}a_{ij}\big|^2}{\frac{|S|}{|S|-1}\sum_{j\in J}\sum_{i\in S}|a_{ij}-\bar a_j|^2},$$

*in words:* $a_{ij}$ is Fourier coefficient $j$ of image $i$, $J$ a ring of frequencies; $s(J)$ is the SSNR of the subset's
average (Section 4, in the paper's form without the −1), $c(J)$ the half-set FSC it predicts, and $r(S)$ sums it over all
rings — a single number that is high when the subset's average is sharp. Choosing among $2^n$ subsets is hopeless by
brute force, so they use **simulated annealing** (as in Module 08's packing, but over subsets):

1. start from a good random pair; 2. propose changes (here: toggle one image in or out of $S$); accept improvements,
   accept a worsening $\Delta r<0$ with the **Metropolis** probability $e^{\Delta r/T_k}$, with temperature $T_k=T_0/(k+1)$
   decreasing with the iteration $k$; 3. after each sweep, **contraction**: remove any image whose removal raises $r$.

⚠️ **A code/paper discrepancy (read before using the official code).** The journal's HTML renders the acceptance
function as $G=\exp(|r(S)-r(S')|/(k+1))$ (the sign of the exponent looks garbled), and AITom's `3d/sa.py` accepts a
worse move when `math.exp(-abs(en - e)/(k+1)) < random.random()` — the *reverse* of the textbook test, which accepts
when $u<e^{-|\Delta|/T}$. We implement the standard Metropolis rule; the function below lets you try the code's rule too.

In [ ]:
# ── 8.1 Data: 40 good + 160 misaligned images of β-galactosidase (pre-aligned stack) ──
def make_stack(n_good: int, n_bad: int, snr: float, rng, rot_sd_deg: float = 5.0, shift_sd: float = 2.0):
    """Good = the reference + noise; bad = slightly misrotated/shifted reference + noise (as in Shi et al.'s simulation)."""
    sigma = np.sqrt(ref2d.var() / snr)
    out, good = [], []
    for i in range(n_good + n_bad):
        img = ref2d if i < n_good else rotate_image(ref2d, np.radians(rng.normal(0, rot_sd_deg)),
                                                    shift_xy=rng.normal(0, shift_sd, 2))
        out.append(img + rng.normal(0, sigma, img.shape)); good.append(i < n_good)
    order = rng.permutation(len(out))
    return np.array(out)[order], np.array(good)[order]


SNR_SEL = 3.0                                     # per-image SNR (Shi et al.'s 2D simulation used SNR 10)
stack, is_good = make_stack(40, 160, SNR_SEL, np.random.default_rng(81))
F_stack = np.fft.fft2(stack).reshape(len(stack), -1)
print(f"{len(stack)} images ({is_good.sum()} good), {S2}×{S2} pixels at {PX2:.2f} Å/pixel, per-image SNR {SNR_SEL}")

In [ ]:
# ── 8.2 The objective r(S) with running sums, and simulated-annealing selection ──
class SubsetScore:
    """Keeps Σ_i a_ij and Σ_i |a_ij|² for the current subset so that r(S) is updated in O(pixels) per move."""

    def __init__(self, F: np.ndarray):
        self.F, self.P = F, np.abs(F) ** 2
        self.sum_F = np.zeros(F.shape[1], complex); self.sum_P = np.zeros(F.shape[1]); self.n = 0

    def toggle(self, i: int, add: bool) -> None:
        """Add (add=True) or remove image i."""
        sgn = 1 if add else -1
        self.sum_F += sgn * self.F[i]; self.sum_P += sgn * self.P[i]; self.n += sgn

    def score(self) -> float:
        """r(S) = Σ_J s/(2+s), s = paper-form SSNR (no −1), DC ring excluded."""
        if self.n < 2:
            return 0.0
        s = ssnr_rings(self.sum_F, self.sum_P, self.n, bias_correct=False)[1:]
        return float((s / (2 + s)).sum())

    def score_if_toggled(self, i: int, add: bool) -> float:
        """r(S) after toggling image i, without keeping the change."""
        self.toggle(i, add); v = self.score(); self.toggle(i, not add)
        return v


def f_beta(selected, good: np.ndarray, beta: float = 0.5):
    """F_β score (β = 0.5 weighs precision more, as in Shi et al.), precision, recall."""
    sel = np.asarray(sorted(selected), int)
    tp = int(good[sel].sum()) if len(sel) else 0
    P, R = tp / max(len(sel), 1), tp / good.sum()
    return (1 + beta ** 2) * P * R / max(beta ** 2 * P + R, 1e-12), P, R


def sa_select(F: np.ndarray, rng, n_sweeps: int = 15, T0: float = 0.05, n_start_pairs: int = 200,
              rule: str = "metropolis", good=None):
    """Simulated-annealing subset selection maximising r(S). rule = 'metropolis' (textbook) or 'aitom' (reverse test)."""
    sc, n = SubsetScore(F), len(F)
    best = None
    for _ in range(n_start_pairs):                                  # 1. a good random starting pair
        a, b = rng.choice(n, 2, replace=False)
        sc.toggle(a, True); sc.toggle(b, True); v = sc.score(); sc.toggle(a, False); sc.toggle(b, False)
        if best is None or v > best[0]:
            best = (v, (a, b))
    inS = np.zeros(n, bool)
    for i in best[1]:
        inS[i] = True; sc.toggle(i, True)
    r = sc.score()
    history = [(r, inS.sum(), f_beta(np.where(inS)[0], good)[0] if good is not None else np.nan)]
    for k in range(n_sweeps):
        T = T0 / (k + 1)                                            # cooling with the iteration index
        for i in rng.permutation(n):                                # 2. propose toggling every image once per sweep
            if inS[i] and inS.sum() <= 2:
                continue
            new = sc.score_if_toggled(i, add=not inS[i])
            d = new - r
            if rule == "metropolis":
                accept = d >= 0 or rng.random() < math.exp(d / T)
            else:                                                    # AITom's test (reverse of Metropolis)
                accept = d >= 0 or math.exp(-abs(d) / (k + 1)) < rng.random()
            if accept:
                sc.toggle(i, add=not inS[i]); inS[i] = ~inS[i]; r = new
        changed = True                                              # 3. contraction
        while changed:
            changed = False
            for i in np.where(inS)[0]:
                if inS.sum() > 2 and sc.score_if_toggled(i, add=False) > r:
                    sc.toggle(i, False); inS[i] = False; r = sc.score(); changed = True
        history.append((r, inS.sum(), f_beta(np.where(inS)[0], good)[0] if good is not None else np.nan))
    return set(np.where(inS)[0]), np.array(history)


t0 = time.time()
sel_sa, hist_sa = sa_select(F_stack, np.random.default_rng(82), good=is_good)
fb_sa, P_sa, R_sa = f_beta(sel_sa, is_good)
print(f"SA: selected {len(sel_sa)} images in {time.time() - t0:.1f} s → precision {P_sa:.2f}, recall {R_sa:.2f}, F0.5 {fb_sa:.2f}")

In [ ]:
# ── 8.3 Baselines: a greedy ranking, random subsets, everything ──
def greedy_ranking(F: np.ndarray, images: np.ndarray, rng, n_start_pairs: int = 200):
    """Our simple greedy baseline: best random pair → rank all images by correlation with the pair's average →
    keep the prefix of the ranking with the highest r(S)."""
    sc, n = SubsetScore(F), len(F)
    best = None
    for _ in range(n_start_pairs):
        a, b = rng.choice(n, 2, replace=False)
        sc.toggle(a, True); sc.toggle(b, True); v = sc.score(); sc.toggle(a, False); sc.toggle(b, False)
        if best is None or v > best[0]:
            best = (v, (a, b))
    avg = images[list(best[1])].mean(0)
    ranked = list(best[1]) + [int(i) for i in np.argsort([-pearson(avg, im) for im in images]) if i not in best[1]]
    scores = []
    for i in ranked:
        sc.toggle(i, True); scores.append(sc.score())
    m = int(np.argmax(scores[1:])) + 2
    return set(ranked[:m])


def r_of(subset) -> float:
    """r(S) of an explicit subset."""
    sc = SubsetScore(F_stack)
    for i in subset:
        sc.toggle(i, True)
    return sc.score()


sel_gr = greedy_ranking(F_stack, stack, np.random.default_rng(83))
sel_rand = set(np.random.default_rng(84).choice(len(stack), len(sel_sa), replace=False))
rows = {"simulated annealing": sel_sa, "greedy ranking": sel_gr, f"random ({len(sel_sa)} images)": sel_rand,
        "all images": set(range(len(stack))), "truly good (oracle)": set(np.where(is_good)[0])}
print(f"{'method':24s} {'|S|':>4s} {'prec':>5s} {'rec':>5s} {'F0.5':>5s} {'r(S)':>6s} {'avg vs ref FRC@0.5':>19s}")
for name, s in rows.items():
    fb, P, R = f_beta(s, is_good)
    f_, c_, _ = fsc(stack[sorted(s)].mean(0), ref2d, PX2)
    print(f"{name:24s} {len(s):4d} {P:5.2f} {R:5.2f} {fb:5.2f} {r_of(s):6.2f} {resolution_at(f_, c_, 0.5):16.1f} Å")

In [ ]:
# ── 8.4 Watch the annealing, and look at the averages ──
fig, axes = plt.subplots(1, 5, figsize=(13, 2.9), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1, 1, 1]))
axes[0].plot(hist_sa[:, 0], "o-", ms=3, label="r(S) of SA subset")
axes[0].axhline(r_of(np.where(is_good)[0]), color="tab:green", ls="--", label="r of the truly good set")
axes[0].set_xlabel("sweep"); axes[0].set_ylabel("r(S)"); axes[0].legend(fontsize=7); axes[0].set_title("objective")
axes[1].plot(hist_sa[:, 2], "s-", ms=3, color="tab:red", label="F0.5")
ax2 = axes[1].twinx(); ax2.plot(hist_sa[:, 1], ":", color="gray"); ax2.set_ylabel("|S| (dotted)", fontsize=8)
axes[1].set_xlabel("sweep"); axes[1].set_ylabel("F0.5"); axes[1].set_ylim(0, 1); axes[1].set_title("selection quality")
for ax, (name, s) in zip(axes[2:], [("all images", rows["all images"]), ("SA selection", sel_sa),
                                    ("truly good", rows["truly good (oracle)"])]):
    ax.imshow(stack[sorted(s)].mean(0)); ax.set_title(f"average: {name}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Starting from a pair, annealing grows the subset while $r(S)$ rises toward the score of the
truly good set; $F_{0.5}$ climbs as the selected images become mostly good ones. The average of the selected images is
visibly sharper than the average of everything, and close to the oracle average. Note that $r(S)$ of the SA subset can
even *exceed* the oracle's: the objective is a noisy estimate, and a few nearly-aligned "bad" images can raise it.

How does the method fare when good particles are rare? The paper reports that SA's advantage over its baselines grows
as the homogeneous fraction shrinks. Let us test our toy at three fractions.

In [ ]:
# ── 8.5 Rare good particles: 10 %, 20 % and 40 % ──
t0 = time.time()
print(f"{'good fraction':>13s} | {'SA F0.5':>7s} {'greedy F0.5':>11s} {'random F0.5':>11s}")
for frac in (0.1, 0.2, 0.4):
    st, gd = make_stack(int(200 * frac), 200 - int(200 * frac), SNR_SEL, np.random.default_rng(int(100 * frac)))
    Fs = np.fft.fft2(st).reshape(len(st), -1)
    s_sa, _ = sa_select(Fs, np.random.default_rng(5), n_sweeps=10)
    s_gr = greedy_ranking(Fs, st, np.random.default_rng(6))
    s_rd = set(np.random.default_rng(7).choice(len(st), len(s_sa), replace=False))
    print(f"{frac:13.1f} | {f_beta(s_sa, gd)[0]:7.2f} {f_beta(s_gr, gd)[0]:11.2f} {f_beta(s_rd, gd)[0]:11.2f}")
print(f"({time.time() - t0:.1f} s)")

🔍 **What just happened?** Both resolution-guided methods are far better than random selection at every fraction. In our
toy the simple greedy ranking is a strong competitor — the misaligned images here differ from the good ones in a single,
simple way. The paper compares SA against its own baselines (greedy image matching "MA" and a genetic algorithm "GA") on
harder simulated and experimental data; our toy does not reproduce that comparison.

✏️ **Your turn.** (1) Lower `SNR_SEL` in 8.1 to `0.5` and re-run 8.1–8.4: where does selection break down? (2) In 8.2 call
`sa_select(..., rule="aitom")` and compare the selected subset with the Metropolis version.

### 📄 Paper connection — Shi *et al.* 2020: simulated annealing for resolution-guided image selection

**Paper**: Shi J, Zeng X, Jiang R, Jiang T, Xu M. *A simulated annealing approach for resolution guided homogeneous
cryo-electron microscopy image selection.* Quantitative Biology 8(1):51–63 (2020).
[doi:10.1007/s40484-019-0191-8](https://doi.org/10.1007/s40484-019-0191-8)

- **Question it answers:** in a pile of noisy pre-aligned 2D images (cryo-EM) or 3D subtomograms (cryo-ET), which subset
  shows the same structure in the same pose — without knowing how many such images exist?
- **What the paper actually does:** maximises the resolution score $r(S)=\sum_J s(J)/(2+s(J))$ (2D SSNR formula above; a
  3D version with missing-wedge masks $m_{ij}$:
  $s(J)=\sum_{j\in J}m_j|\bar a_j|^2\big/\sum_{j\in J}\frac{1}{m_j-1}\sum_{i\in S}m_{ij}|a_{ij}-\bar a_j|^2$) by simulated
  annealing over subsets: start from a random pair above a score threshold, propose adding/replacing images, accept
  worse moves with an iteration-dependent probability, and apply a **contraction** step after each sweep; **SA+**
  restarts from two unselected images and merges non-overlapping selections, fixing stalls on 3D data. Baselines: MA
  (greedy image matching from the best pair) and GA (genetic algorithm over inclusion vectors). Metrics: $F_{0.5}$,
  precision, recall, resolution score, time.
- **What our toy keeps:** the objective $r(S)$ in the paper's form, SA over subsets with contraction, $F_{0.5}$ evaluation,
  misaligned "bad" images (rotation and shift errors, as in the paper's 2D simulation).
- **What our toy leaves out:** 3D subtomograms with wedge masks, SA+, the MA and GA baselines as published, the
  experimental *P. falciparum* ribosome images and ribosome subtomograms. We use the **standard Metropolis rule**, not
  the code's reversed test (see the warning above).
- **Result reported:** on 2D data the authors report that SA beats MA and GA in $F_{0.5}$ and resolution score at lower
  cost than GA and converges in about 10 iterations, with a larger advantage when the homogeneous fraction is small; on 3D
  data plain SA can stall, and SA+ beats SA and GA (table values not extracted in our notes).
- **Code:** AITom `aitom/classify/select/simulated_annealing/` (`2d/`, `3d/`: `sa.py`, `sa_restart.py`, `ga.py`, `ma.py`,
  `ssnr2d.py`/`ssnr3d.py`; Python 2.7), tutorial pointer `doc/tutorials/018_simulated_annealing.md`.

---
## ✅ Checkpoint

1. A map is sampled at 1.3 Å/pixel. What is the best resolution it could possibly show?
   <details><summary>Answer</summary>The Nyquist limit, 2 × 1.3 = 2.6 Å.</details>
2. Which FSC threshold do you use for (a) two gold-standard half-maps, (b) a simulated map against its known ground truth? Why do they correspond?
   <details><summary>Answer</summary>(a) 0.143, (b) 0.5. If FSC_half = 0.143, the full map's correlation with the truth is C_ref = √(2·0.143/1.143) = 0.5 (Rosenthal & Henderson 2003).</details>
3. Each half-map in a shell has SSNR 1. What FSC do you expect between the half-maps? What is the SSNR η of the full set, and what does η/(2+η) give?
   <details><summary>Answer</summary>FSC = 1/(1+1) = 0.5. The full set has η = 2, and η/(2+η) = 2/4 = 0.5 — the same number, as it must be.</details>
4. Your FSC curve stays above 0.4 all the way to Nyquist. Name two things to check before believing it.
   <details><summary>Answer</summary>The mask (sharp/tight masks inflate high-frequency FSC — check with phase randomisation) and the independence of the half-maps (were they refined separately from a low-pass-filtered start? any shared reference or particles?).</details>
5. Why is local resolution computed in windows of limited size, and what is the trade-off?
   <details><summary>Answer</summary>To see spatial variation (core vs flexible periphery). Small windows localise better but have few Fourier coefficients per shell, so their FSC estimates are noisy; large windows are precise but blur spatial detail.</details>
6. In simulated annealing at temperature T = 0.01, a toggle lowers r(S) by 0.02. With what probability does the Metropolis rule accept it?
   <details><summary>Answer</summary>e^(−0.02/0.01) = e^(−2) ≈ 0.14.</details>

## 🧠 Questions & discussion

1. FSC measures *reproducibility* between halves, not *correctness*. Describe a situation where both half-maps agree
   perfectly but the map is wrong (hint: Section 3.2, or a mixture of conformations).
2. Why might a flexible domain look worse in the local-resolution map even if it is perfectly ordered in each individual
   molecule? (Think about heterogeneity, Module 13.)
3. Shi *et al.*'s objective uses all frequency rings equally. What could go wrong if most rings contain only noise? How
   would you modify $r(S)$?
4. MPP (Module 14) predicts half-set FSC from SSNR instead of splitting. When is that cheaper, and what assumption about
   the members must hold?
5. In cryo-ET, every subtomogram misses a wedge of Fourier space. How must the SSNR (and FSC) computation change? (Hint:
   the masks $m_{ij}$ in Shi's 3D formula and $\tilde M$ in MPP's.)

## 🚀 Bonus — going further

- **Real half-maps:** download the half-maps of an EMDB entry (many entries deposit them), compute the FSC with the
  toolkit's `fsc`, and compare with the deposited curve.
- **½-bit criterion:** implement the van Heel & Schatz (2005) threshold curve (it depends on the number of voxels per
  shell) and compare with 0.143 on our half-maps.
- **3D selection:** extend `SubsetScore` to subtomograms with missing-wedge masks using Shi's 3D formula (count how many
  subtomograms observe each coefficient) and test it on Module 08's simulated ribosomes.
- **Real Spark:** run the lab's code at <https://github.com/xulabs/projects> (or `pyspark` locally) on our half-maps.
- **Deep dives in this repo:** [`../../LocalResolution/`](../../LocalResolution/) and
  [`../../SubtomogramAveraging/`](../../SubtomogramAveraging/).

## 📝 Lab note

```text
Module 09 — date:
Input: β-galactosidase map (NGL test data, 3.54 Å/voxel) and its local-resolution map
Output: half-map FSC curves, masked/unmasked comparisons, Einstein-from-noise FRC, SSNR→FSC check,
        FSC vs N, local-resolution map, parallel timing, SA-selected particle subsets
Method: FSC (0.143 half-maps / 0.5 vs truth), phase randomisation, SSNR, sliding-window FSC, Metropolis SA
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Lü et al. 2020 (Spark local FSC); Shi et al. 2020 (SA selection); Xu et al. 2019 (SFSC)
```

## 📚 References

1. Lü Y, Zeng X, Tian X, Shi X, Wang H, Zheng X, Liu X, Zhao X, Gao X, Xu M. Spark-based parallel calculation of 3D Fourier shell correlation for macromolecule structure local resolution estimation. *BMC Bioinformatics* 21(Suppl 13):391 (2020). doi:10.1186/s12859-020-03680-6
2. Shi J, Zeng X, Jiang R, Jiang T, Xu M. A simulated annealing approach for resolution guided homogeneous cryo-electron microscopy image selection. *Quant Biol* 8(1):51–63 (2020). doi:10.1007/s40484-019-0191-8
3. Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F. De novo structural pattern mining in cellular electron cryotomograms. *Structure* 27(4):679–691 (2019). doi:10.1016/j.str.2019.01.005
4. Rosenthal PB, Henderson R. Optimal determination of particle orientation, absolute hand, and contrast loss in single-particle electron cryomicroscopy. *J Mol Biol* 333:721–745 (2003). doi:10.1016/j.jmb.2003.07.013
5. van Heel M, Schatz M. Fourier shell correlation threshold criteria. *J Struct Biol* 151:250–262 (2005). doi:10.1016/j.jsb.2005.05.009
6. Scheres SHW, Chen S. Prevention of overfitting in cryo-EM structure determination. *Nat Methods* 9:853–854 (2012). doi:10.1038/nmeth.2115
7. Chen S, McMullan G, Faruqi AR, Murshudov GN, Short JM, Scheres SHW, Henderson R. High-resolution noise substitution to measure overfitting and validate resolution in 3D structure determination by single particle electron cryomicroscopy. *Ultramicroscopy* 135:24–35 (2013). doi:10.1016/j.ultramic.2013.06.004
8. Henderson R. Avoiding the pitfalls of single particle cryo-electron microscopy: Einstein from noise. *PNAS* 110:18037–18041 (2013). doi:10.1073/pnas.1314449110
9. Unser M, Trus BL, Steven AC. A new resolution criterion based on spectral signal-to-noise ratios. *Ultramicroscopy* 23:39–51 (1987). doi:10.1016/0304-3991(87)90225-7
10. Penczek PA. Three-dimensional spectral signal-to-noise ratio for a class of reconstruction algorithms. *J Struct Biol* 138:34–46 (2002). doi:10.1016/S1047-8477(02)00033-3
11. Cardone G, Heymann JB, Steven AC. One number does not fit all: mapping local variations in resolution in cryo-EM reconstructions. *J Struct Biol* 184:226–236 (2013). doi:10.1016/j.jsb.2013.08.002
12. Kucukelbir A, Sigworth FJ, Tagare HD. Quantifying the local resolution of cryo-EM density maps. *Nat Methods* 11:63–65 (2014). doi:10.1038/nmeth.2727
13. Kirkpatrick S, Gelatt CD, Vecchi MP. Optimization by simulated annealing. *Science* 220:671–680 (1983). doi:10.1126/science.220.4598.671
14. NGL viewer test data (β-galactosidase map and local-resolution map), <https://github.com/nglviewer/ngl> (MIT licence).

## ➡️ Next

**Part C begins: [10 · Finding particles: template matching, difference-of-Gaussians and their pitfalls](10_particle_picking_classical.ipynb)**
— with simulated tomograms (Module 08) and a way to judge quality (this module), we are ready to find molecules in a cell.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")